# Lab 4 — Two desks, one queue, one decision

**~25 minutes · nothing to fill in · Run All takes about 3 minutes**

You have now built the Global Bank support desk twice: as a crew in Labs 1 and 2, and in ADK in Lab 3.
This lab builds the best version of each one:

- the **crew** with the bank's records and the Lab 2 guardrail, and
- the **ADK desk** with the bank's records.

Both handle the same five tickets. The same checks score both. Then you choose one for your own team's
work, and say what would change your mind.

That last step is the real work of this lab. The scorecard helps you check your argument.

## 1 · One queue, one set of checks

Both desks get the same tickets, the same two tools, the same three roles and the same two record
shapes. **Only the framework changes.** That is the only way the comparison means anything.

In [ ]:
import os, time
from desk_kit import (TICKETS, ACCOUNT_ACTIVITY, TEAM_RULE, check_reply, print_queue, score,
                      EXPECTED_TEAM, is_grounded)

MODEL = os.environ["OPENAI_MODEL"]
BASE  = os.environ["OPENAI_BASE_URL"]
KEY   = os.environ["OPENAI_API_KEY"]

from pydantic import BaseModel, Field

class Triage(BaseModel):
    category: str = Field(description="The kind of problem, in a few words")
    team: str = Field(description="The team that owns the ticket")
    priority: str = Field(description="High, Medium or Low")

class DeskReply(BaseModel):
    customer_reply: str = Field(description="The reply the customer reads, at most four sentences")
    summary: str = Field(description="Handover: what happened, in one sentence")
    what_to_check: str = Field(description="Handover: what the owning team should check first")
    next_action: str = Field(description="Handover: the next thing the owning team should do")

RESULTS = {}

def record(name, rows, seconds, requests, tokens):
    RESULTS[name] = {"rows": rows, "seconds": round(seconds), "requests": requests, "tokens": tokens,
                     **score(rows)}

def scoreboard():
    cols = ["right team", "safe reply", "says what happened", "handover", "seconds", "requests"]
    print(f"{'desk':<16}" + "".join(f"{c:>20}" for c in cols))
    for name, r in RESULTS.items():
        print(f"{name:<16}" + "".join(f"{str(r[c]):>20}" for c in cols))

print(len(TICKETS), "tickets in the queue")

## 2 · The crew

This is the Lab 2 desk with two changes: the researcher also has `account_activity`, and the writer's
task keeps the guardrail.

In [ ]:
from crewai import LLM, Agent as CrewAgent, Task, Crew, Process
from crewai.tools import tool

@tool("ticket_lookup")
def crew_ticket_lookup(ticket_id: str) -> str:
    """Return the text of a Global Bank customer support ticket by its id, e.g. GB-T-4471."""
    return TICKETS.get(ticket_id, "No ticket found")

@tool("account_activity")
def crew_account_activity(ticket_id: str) -> str:
    """Return what the bank's systems show for the account in a ticket: payments, fees and requests."""
    return ACCOUNT_ACTIVITY.get(ticket_id, "No records found")

crew_llm = LLM(model=MODEL, base_url=BASE, api_key=KEY, temperature=0)   # plain name

def reply_is_safe(output):
    problems = check_reply(output.pydantic.customer_reply)
    return (False, " ".join(problems) + " Rewrite the customer reply.") if problems else (True, output)

researcher = CrewAgent(role="Ticket Researcher", goal="State the facts in the ticket and the account records",
                       backstory="You pull the raw ticket and the account records. You never guess.",
                       llm=crew_llm, tools=[crew_ticket_lookup, crew_account_activity], allow_delegation=False)
classifier = CrewAgent(role="Support Triage Analyst", goal="Classify a ticket, name the owning team and set a priority",
                       backstory="You triage the Global Bank customer support desk.",
                       llm=crew_llm, allow_delegation=False)
writer = CrewAgent(role="Response Drafter", goal="Write the customer reply and the handover note",
                   backstory="You write to Global Bank customers. You tell them what the records show, "
                             "promise only what the team can do, and never invent a timeline.",
                   llm=crew_llm, allow_delegation=False)

crew = Crew(
    agents=[researcher, classifier, writer],
    tasks=[Task(description="Retrieve ticket {ticket_id} and its account records, and list the facts.",
                expected_output="A short bulleted list of facts.", agent=researcher),
           Task(description=f"Classify ticket {{ticket_id}} and name the owning team. {TEAM_RULE} "
                            "The priority is High if the customer has lost money, otherwise Medium or Low.",
                expected_output="The category, the team and the priority.", agent=classifier,
                output_pydantic=Triage),
           Task(description="Write the first reply to the customer who sent ticket {ticket_id}, "
                            "and a handover note for the owning team.",
                expected_output="The customer reply and the three parts of the handover note.",
                agent=writer, output_pydantic=DeskReply,
                guardrail=reply_is_safe, guardrail_max_retries=2)],
    process=Process.sequential, verbose=False)

rows = []
before = crew_llm.get_token_usage_summary()
start = time.time()
for ticket_id in TICKETS:
    out = await crew.kickoff_async(inputs={"ticket_id": ticket_id})
    decision, reply = crew.tasks[1].output.pydantic, out.pydantic
    rows.append({"ticket": ticket_id, "team": decision.team, "priority": decision.priority,
                 "reply": reply.customer_reply, "summary": reply.summary,
                 "what_to_check": reply.what_to_check, "next_action": reply.next_action})
used = crew_llm.get_token_usage_summary().delta_since(before)
record("CrewAI", rows, time.time() - start, used.successful_requests, used.total_tokens)

print_queue(rows)

## 3 · The ADK desk

This is the Lab 3 desk with the records. `run_adk_desk` has one setting, `include_contents`. Section 4
uses it.

In [ ]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

def adk_model():
    return LiteLlm(model=f"openai/{MODEL}", api_base=BASE, api_key=KEY)   # prefix + api_base

def ticket_lookup(ticket_id: str) -> dict:
    """Return the text of a Global Bank customer support ticket by its id, e.g. GB-T-4471."""
    return {"ticket_id": ticket_id, "text": TICKETS.get(ticket_id, "No ticket found")}

def account_activity(ticket_id: str) -> dict:
    """Return what the bank's systems show for the account in a ticket: payments, fees and requests."""
    return {"ticket_id": ticket_id, "records": ACCOUNT_ACTIVITY.get(ticket_id, "No records found")}

RULES = ("Never promise a refund or credit, never give a deadline, "
         "and never ask for an OTP, PIN, CVV or password.")

async def run_adk_desk(name, include_contents="default"):
    researcher = Agent(name="researcher", model=adk_model(), tools=[ticket_lookup, account_activity],
                       instruction="Use your tools to look up everything about the ticket. "
                                   "List only the facts. Do not interpret them.",
                       output_key="facts")
    classifier = Agent(name="classifier", model=adk_model(), output_schema=Triage, output_key="triage",
                       include_contents=include_contents,
                       instruction="Facts:\n{facts}\n\nClassify the ticket. " + TEAM_RULE +
                                   " The priority is High if the customer has lost money, "
                                   "otherwise Medium or Low.")
    drafter = Agent(name="drafter", model=adk_model(), output_schema=DeskReply, output_key="reply",
                    include_contents=include_contents,
                    instruction="Facts:\n{facts}\nTriage:\n{triage}\n\nWrite the first reply to the "
                                "customer, and a handover note for the owning team. "
                                "Tell the customer what the facts show. " + RULES)

    svc = InMemorySessionService()
    runner = Runner(app_name="cmp", session_service=svc,
                    agent=SequentialAgent(name="desk", sub_agents=[researcher, classifier, drafter]))
    rows, requests, tokens = [], 0, 0
    start = time.time()
    for ticket_id in TICKETS:
        await svc.create_session(app_name="cmp", user_id="u", session_id=ticket_id)
        msg = types.Content(role="user", parts=[types.Part(text=f"Handle ticket {ticket_id}.")])
        async for ev in runner.run_async(user_id="u", session_id=ticket_id, new_message=msg):
            if ev.usage_metadata:                           # one per model call
                requests += 1
                tokens += ev.usage_metadata.total_token_count or 0
        state = (await svc.get_session(app_name="cmp", user_id="u", session_id=ticket_id)).state
        decision, reply = state["triage"], state["reply"]
        rows.append({"ticket": ticket_id, "team": decision["team"], "priority": decision["priority"],
                     "reply": reply["customer_reply"], "summary": reply["summary"],
                     "what_to_check": reply["what_to_check"], "next_action": reply["next_action"]})
    record(name, rows, time.time() - start, requests, tokens)
    return rows

adk_rows = await run_adk_desk("ADK")
print_queue(adk_rows)
print()
scoreboard()

## 4 · Decide what each agent receives

By default, an ADK sub-agent receives the **whole session so far**: the customer's message, the
researcher's tool calls and their results, and every earlier answer. It *also* gets the state you put
into its instruction, so `{facts}` arrives twice.

`include_contents="none"` sends only the instruction, with the state you named in it. The agent reads
less, and every model call is smaller.

**Predict before you run it:** will the drafter still tell the customer what happened, when all it sees
is `{facts}` and `{triage}`?

In [ ]:
await run_adk_desk("ADK, none", include_contents="none")
scoreboard()
print()
for name, r in RESULTS.items():
    print(f"{name:<12} {r['tokens']:>7} tokens")

**Check the quality before you keep a saving.** Compare the `says what happened` column for the two ADK
rows. If it dropped, the drafter lost detail that it had found in the tool results. The researcher's
list of facts did not carry every detail.

That is the rule for any change that makes an agent read less: **cut, then check that the quality
held.** Here the check takes seconds, because you have a scorecard.

## 5 · Read the hard ticket, three ways

`GB-T-4475` demands a refund by tomorrow. Read how each desk answered it, as the customer would, and
then as the Transactions team would.

In [ ]:
for name, r in RESULTS.items():
    row = next(x for x in r["rows"] if x["ticket"] == "GB-T-4475")
    print("=" * 72)
    print(name, "- team:", row["team"], "| safe:", check_reply(row["reply"]) or "yes")
    print("=" * 72)
    print("REPLY   :", row["reply"])
    print("HANDOVER:", row["summary"])
    print("          check:", row["what_to_check"])
    print("          next :", row["next_action"])
    print()

## 6 · Where the differences come from

Be clear about *why* the desks differ before you conclude anything. On the same model, neither
framework is smarter. The differences come from what each one does by default.

- **Checking a reply.** The crew has a guardrail: an unsafe reply is sent back and rewritten. The ADK
  desk has only the rules in its instruction. It usually keeps them, but nothing checks. In ADK you add
  the check yourself, in an `after_agent_callback` or in your own code after the run.
- **What each agent reads.** CrewAI sends each agent its role, goal and backstory, the task, and the
  outputs of the tasks before it. ADK by default sends the whole session. With `include_contents="none"`
  it sends only what you named.
- **Seeing the path.** ADK gave you every tool call in the event stream. The crew hid them unless you
  turn on logging.
- **The model varies.** Run the lab again. A column can move by one ticket on a second run. Five tickets
  show a direction, not a final score.

## 7 · Choose one

Answer these questions for a real piece of work in your team. Write the answers down. Writing them is
the point of this section.

1. **Do you know the order of the steps in advance?** If yes, neither framework's dynamic routing helps
   you. A graph or a plain function might be the better answer.
2. **What must never reach a customer or a system?** If you have rules like the bank's, where does each
   framework let you enforce them: a guardrail, a callback, or your own code?
3. **Who needs to read this code in six months?** Roles and backstories read like job descriptions.
   Explicit runners and named state read like a program. Your team has a preference, and it matters.
4. **How will you know the desk is still right next month?** You now have a queue with known answers
   and a scorecard. Which framework makes it easier to run that check after every change?
5. **What single fact would change your mind?** If you cannot name one, you have a preference, not a
   decision.

### What each framework gives you

| | CrewAI | ADK |
|---|---|---|
| **Setup for a small job** | Low: agents, tasks, crew, then run | Higher: runner, session, event loop |
| **Structured answers** | `output_pydantic` on a task | `output_schema` on an agent (no tools on that agent) |
| **Checking an answer** | `guardrail` on a task, with retries built in | Instructions, plus a callback or code you write |
| **Hand-offs** | Earlier task outputs, passed as context | `output_key` and `{name}`, visible in the code |
| **What each call reads** | Role, goal, backstory and earlier task outputs | Whole session by default, or only what you named |
| **Seeing the path** | Hidden unless you turn on logging | Every tool call is an event you can read |
| **Running in a notebook** | `kickoff()` fails in Jupyter. Use `kickoff_async()` | Async first. No wrong version to call |

**Neither framework is the advanced one.** Pick the one that makes your job easier to get right, and be
able to say why.

---

**You can now** put two frameworks on one real queue, score their work with the same checks, and defend
your choice with what each desk actually delivered.